In [1]:
import pandas as pd
import numpy as np
import re
from collections import Counter

INPUT_CSV = "cards.csv"
OUTPUT_CSV = "cards_cleaned123.csv"
OVERWRITE_INPUT = False
df = pd.read_csv(INPUT_CSV, low_memory=False)

In [2]:
# Missing Values

invalid_placeholders = {"", " ", "n/a", "na", "nan", "none", "null", "?", "unknown", "missing"}

invalid_mask = df.applymap(
    lambda x: (
        pd.isna(x) or
        (isinstance(x, str) and x.strip().lower() in invalid_placeholders)
    )
)

# Anzahl ungültiger Werte pro Spalte
invalid_counts = invalid_mask.sum()

# Gesamtanzahl Einträge pro Spalte
total_counts = len(df)

# Gesamtsumme aller ungültigen Werte
total_invalid = invalid_counts.sum()

print("Übersicht fehlender oder ungültiger Werte")
for col in df.columns:
    count = invalid_counts[col]
    if count > 0:
        print(f"{col}: {count}/{total_counts} ungültig ({count / total_counts:.2%})")


print(f"Gesamt ungültige Werte im Datensatz: {total_invalid}/{df.size} ({total_invalid / df.size:.2%})")

Übersicht fehlender oder ungültiger Werte
volatility: 2659/40792 ungültig (6.52%)
sub_type: 7831/40792 ungültig (19.20%)
attribute: 14838/40792 ungültig (36.37%)
rank: 14838/40792 ungültig (36.37%)
attack: 15135/40792 ungültig (37.10%)
defense: 15033/40792 ungültig (36.85%)
set_release: 53/40792 ungültig (0.13%)
Gesamt ungültige Werte im Datensatz: 70387/775048 (9.08%)


In [3]:
print("Input shape:", df.shape)
print("Spalten:", list(df.columns))

# Übersicht fehlende Werte (absolute + Prozent)
missing = df.isna().sum().sort_values(ascending=False)
missing_pct = (df.isna().mean() * 100).round(2).sort_values(ascending=False)
print("\n Fehlende Werte (absteigend)")
print(pd.concat([missing, missing_pct], axis=1).rename(columns={0:"missing_count", 1:"missing_pct"}).head(50))

# Beispielzeilen mit vielen NaNs — wichtig um Problemtypen zu sehen
df['missing_count'] = df.isna().sum(axis=1)
#print("\nBeispiele: Zeilen mit den meisten fehlenden Werten:")
#display(df.sort_values('missing_count', ascending=False).head(10).T)
df = df.drop(columns=['missing_count'])

Input shape: (40792, 19)
Spalten: ['Unnamed: 0', 'name', 'description', 'set_id', 'rarity', 'price', 'volatility', 'type', 'sub_type', 'attribute', 'rank', 'attack', 'defense', 'set_name', 'set_release', 'name_official', 'index', 'index_market', 'join_id']

 Fehlende Werte (absteigend)
               missing_count  missing_pct
attribute              14838        36.37
attack                 14838        36.37
defense                14838        36.37
rank                   14838        36.37
sub_type                7831        19.20
volatility              2659         6.52
set_release               53         0.13
index_market               0         0.00
index                      0         0.00
name_official              0         0.00
set_name                   0         0.00
Unnamed: 0                 0         0.00
name                       0         0.00
type                       0         0.00
price                      0         0.00
rarity                     0         0.00

In [4]:
# Die Analyse zeigt, dass die meisten fehlenden Werte im Yu-Gi-Oh!-Datensatz logisch erklärbar und problemlos sind, 
# da sie nur bei bestimmten Kartentypen (z. B. Nicht-Monstern) auftreten.
# Lediglich die Spalte „volatility“ sollte genauer geprüft werden, da hier die fehlenden Werte auf unvollständige Preisdaten hindeuten könnten.

In [5]:
print(df.dtypes)

Unnamed: 0        int64
name             object
description      object
set_id           object
rarity           object
price            object
volatility       object
type             object
sub_type         object
attribute        object
rank             object
attack           object
defense          object
set_name         object
set_release      object
name_official    object
index             int64
index_market      int64
join_id          object
dtype: object


In [6]:
import pandas as pd
import numpy as np
import logging

# Hilfsfunktion: sichere numerische Konvertierung
def to_numeric_safe(series):
    return pd.to_numeric(series, errors='coerce')

# Numerische Spalten konvertieren
num_candidates = ['price', 'attack', 'defense', 'rank', 'volatility']
for col in num_candidates:
    if col in df.columns:
        if col == 'volatility':
            # evtl. '%' entfernen und 'nan' durch np.nan ersetzen
            df[col] = df[col].astype(str).str.replace('%', '', regex=False).replace({'nan': np.nan})
        df[col] = to_numeric_safe(df[col])


# Datums-Spalte konvertieren
if 'set_release' in df.columns:
        df['set_release_parsed'] = pd.to_datetime(df['set_release'], errors='coerce')

# Kategorische Felder vereinheitlichen
cat_cols = ['sub_type','attribute','rarity','set_name','set_id','name_official']
for c in cat_cols:
    if c in df.columns:
        df[c] = df[c].astype(str).str.strip().replace({'nan': np.nan, 'None': np.nan})



In [7]:
num_duplicates = df.duplicated().sum()
print(f"Anzahl exakter Duplikate: {num_duplicates}")

Anzahl exakter Duplikate: 0


In [8]:
print(df["type"].unique())


['SPELL' 'MONSTER' 'TRAP']


In [9]:
df.to_csv(OUTPUT_CSV, index=False)
if OVERWRITE_INPUT:
    df.to_csv(INPUT_CSV, index=False)

print(f"Bereinigte CSV gespeichert nach: {OUTPUT_CSV}")

Bereinigte CSV gespeichert nach: cards_cleaned123.csv


In [15]:
df1 = pd.read_csv(OUTPUT_CSV, low_memory=False)

In [16]:
print(df1["type"].unique())

['SPELL' 'MONSTER' 'TRAP']


In [11]:
'''# Duplikat-Analyse
# Anzahl verschiedener Namen
unique_names = df['name'].nunique()
total_rows = len(df)
print(f"Total rows: {total_rows}, unique names: {unique_names}")

#Kartenname mit Mehrfachvorkommen (nur Name)
name_counts = df['name'].value_counts()
dupe_names = name_counts[name_counts > 1]
print("\nNamen die mehrfach vorkommen (Top 20):")
print(dupe_names.head(20))

# Echte Duplikate: gleiche spielrelevante Attribute
dup_key_cols = ['name', 'description', 'type', 'sub_type', 'attribute', 'attack', 'defense']
# Nur vorhandene Spalten nutzen
dup_key_cols = [c for c in dup_key_cols if c in df.columns]
duplicates_full = df.duplicated(subset=dup_key_cols, keep=False)
print("\nAnzahl Zeilen, die echte Duplikate (gleich in allen spielrelevanten Attributen):", duplicates_full.sum())
'''

'# Duplikat-Analyse\n# Anzahl verschiedener Namen\nunique_names = df[\'name\'].nunique()\ntotal_rows = len(df)\nprint(f"Total rows: {total_rows}, unique names: {unique_names}")\n\n#Kartenname mit Mehrfachvorkommen (nur Name)\nname_counts = df[\'name\'].value_counts()\ndupe_names = name_counts[name_counts > 1]\nprint("\nNamen die mehrfach vorkommen (Top 20):")\nprint(dupe_names.head(20))\n\n# Echte Duplikate: gleiche spielrelevante Attribute\ndup_key_cols = [\'name\', \'description\', \'type\', \'sub_type\', \'attribute\', \'attack\', \'defense\']\n# Nur vorhandene Spalten nutzen\ndup_key_cols = [c for c in dup_key_cols if c in df.columns]\nduplicates_full = df.duplicated(subset=dup_key_cols, keep=False)\nprint("\nAnzahl Zeilen, die echte Duplikate (gleich in allen spielrelevanten Attributen):", duplicates_full.sum())\n'

In [12]:
'''# Definiere die Gruppierung nach spielrelevanten Attributen
group_key = ['name', 'description', 'type', 'sub_type', 'attribute', 'attack', 'defense']
group_key = [c for c in group_key if c in df.columns]

# Aggregationslogik
agg_dict = {}

# Numerische Spalten -> Median
numeric_cols = ['price', 'volatility', 'rank']  
for c in numeric_cols:
    if c in df.columns:
        agg_dict[c] = 'median'

# Set-bezogene Spalten -> Liste
if 'set_id' in df.columns:
    agg_dict['set_id'] = lambda x: list(x.astype(str))

# Andere Spalten -> First
for c in df.columns:
    if c not in group_key and c not in agg_dict:
        agg_dict[c] = 'first'

# Gruppieren
df_grouped = df.groupby(group_key, as_index=False).agg(agg_dict)

# Neue Spalte "Decks" erzeugen (z. B. Kombination aus set_id + set_name)
if 'set_id' in df.columns and 'set_name' in df.columns:
    def build_deck(row):
        ids = row['set_id']
        names = df.loc[df['name'] == row['name'], 'set_name'].unique().tolist()
        return [f"{i} ({n})" for i, n in zip(ids, names[:len(ids)])]

    df_grouped['Decks'] = df_grouped.apply(build_deck, axis=1)

# Datum konvertieren
if 'set_release' in df_grouped.columns:
    df_grouped['set_release'] = pd.to_datetime(df_grouped['set_release'], errors='coerce')

print(df_grouped[decks].head(1))
'''

'# Definiere die Gruppierung nach spielrelevanten Attributen\ngroup_key = [\'name\', \'description\', \'type\', \'sub_type\', \'attribute\', \'attack\', \'defense\']\ngroup_key = [c for c in group_key if c in df.columns]\n\n# Aggregationslogik\nagg_dict = {}\n\n# Numerische Spalten -> Median\nnumeric_cols = [\'price\', \'volatility\', \'rank\']  \nfor c in numeric_cols:\n    if c in df.columns:\n        agg_dict[c] = \'median\'\n\n# Set-bezogene Spalten -> Liste\nif \'set_id\' in df.columns:\n    agg_dict[\'set_id\'] = lambda x: list(x.astype(str))\n\n# Andere Spalten -> First\nfor c in df.columns:\n    if c not in group_key and c not in agg_dict:\n        agg_dict[c] = \'first\'\n\n# Gruppieren\ndf_grouped = df.groupby(group_key, as_index=False).agg(agg_dict)\n\n# Neue Spalte "Decks" erzeugen (z. B. Kombination aus set_id + set_name)\nif \'set_id\' in df.columns and \'set_name\' in df.columns:\n    def build_deck(row):\n        ids = row[\'set_id\']\n        names = df.loc[df[\'name\'

In [13]:
'''df_grouped.to_csv(OUTPUT_CSV, index=False)
if OVERWRITE_INPUT:
    df_grouped.to_csv(INPUT_CSV, index=False)
print(f"\nBereinigte CSV gespeichert nach: {OUTPUT_CSV}")
'''

'df_grouped.to_csv(OUTPUT_CSV, index=False)\nif OVERWRITE_INPUT:\n    df_grouped.to_csv(INPUT_CSV, index=False)\nprint(f"\nBereinigte CSV gespeichert nach: {OUTPUT_CSV}")\n'

In [14]:
'''df_filtered = df[df['name'] == "Monster Rei"] '''

'df_filtered = df[df[\'name\'] == "Monster Rei"] '